# NBA MVP Voting Data
This is a notebook that will create a dataset that has all of the NBA players in MVP voting from 2000 to 2025.

Source: [Basketball Reference](https://www.basketball-reference.com)

Authors: Rasheed Martin

## Import relevant packages

In [157]:
from pathlib import Path
import polars as pl

## Combining Multiple CSVS into DataFrame

In this step we have already manually downloaded all the csvs from @ [Basketball Reference](https://www.basketball-reference.com). 
We had the option of webscrapping but we were having API issues so we pivoted. 

In [158]:
# Schema Definition
schema = {
    'Rank': pl.String, 'Player': pl.String, 'Age': pl.Int64, 'Tm': pl.String, 'First': pl.Int64, 'Pts Won': pl.Int64, 'Pts Max': pl.Int64, 'Share': pl.Float64, 'G': pl.Int64, 'MP': pl.Float64, 'PTS': pl.Float64, 'TRB': pl.Float64, 'AST': pl.Float64, 'STL': pl.Float64, 'BLK': pl.Float64, 'FG%': pl.Float64, '3P%': pl.Float64, 'FT%': pl.Float64, 'WS': pl.Float64, 'WS/48': pl.Float64, '-9999': pl.String}

# Loop over CSVs path and convert to individual dataframes
df_list = [pl.read_csv(path, schema=schema) for path in Path("data/raw_data/MVP_Voting/raw_csvs").iterdir()]

# Add the dataframes together vertically (more rows, matching columns) = More records
master_df = pl.concat(df_list, how="vertical")
master_df.head()

Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,-9999
str,str,i64,str,i64,i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
"""1""","""Tim Duncan""",26,"""SAS""",60,962,1190,0.808,81,39.3,23.3,12.9,3.9,0.7,2.9,0.513,0.273,0.71,16.5,0.248,"""duncati01"""
"""2""","""Kevin Garnett""",26,"""MIN""",43,871,1190,0.732,82,40.5,23.0,13.4,6.0,1.4,1.6,0.502,0.282,0.751,15.6,0.225,"""garneke01"""
"""3""","""Kobe Bryant""",24,"""LAL""",8,496,1190,0.417,82,41.5,30.0,6.9,5.9,2.2,0.8,0.451,0.383,0.843,14.9,0.21,"""bryanko01"""
"""4""","""Tracy McGrady""",23,"""ORL""",4,427,1190,0.359,75,39.4,32.1,6.5,5.5,1.7,0.8,0.457,0.386,0.793,16.1,0.262,"""mcgratr01"""
"""5""","""Shaquille O'Neal""",30,"""LAL""",3,126,1190,0.106,67,37.8,27.5,11.1,3.1,0.6,2.4,0.574,null,0.622,13.2,0.25,"""onealsh01"""


## Clean the Data

Here we slightly clean the data by dropping irrelevant columns and fixing column names

In [159]:
master_df.columns

['Rank',
 'Player',
 'Age',
 'Tm',
 'First',
 'Pts Won',
 'Pts Max',
 'Share',
 'G',
 'MP',
 'PTS',
 'TRB',
 'AST',
 'STL',
 'BLK',
 'FG%',
 '3P%',
 'FT%',
 'WS',
 'WS/48',
 '-9999']

### Drop Unknown column and Voting Information

In [160]:
master_df = master_df.drop(pl.col("-9999"), strict=False)
master_df = master_df.drop(["Pts Won", "Pts Max", "Share", "First"], strict=False)

### Rename some of the columns to clearer names

In [161]:
master_df = master_df.rename({
    "Tm": "Team_Abbreviation", 
    "G": "Games",
    "MP": "Minutes_Per_Game",
    "PTS": "Points_Per_Game",
    "TRB": "Total_Rebounds_Per_Game",
    "AST": "Assists_Per_Game",
    "STL": "Steals_Per_Game",
    "BLK": "Blocks_Per_Game",
    "FG%": "Field_Goal_Percentage",
    "3P%": "3-Points_Field_Goal_Percentage",
    "FT%": "Free_Throw_Percentage",
    "WS": "Win_Shares",
    "WS/48": "Win_Shares_Per_48_Minutes"
})

### Remove the "T" from Rank


In [162]:
master_df = master_df.with_columns(
    pl.col("Rank").str.replace("T", "").cast(pl.Int64)
)
master_df["Rank"].unique().to_list()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

## Fix the Data Quality

### Change the Team Abbreviation to Full Name.

In [163]:
from utils.utils import nba_teams

master_df = master_df.with_columns(
    pl.col("Team_Abbreviation").map_elements(lambda x: nba_teams.get(x, "Unknown"), return_dtype=pl.String).alias("Team_Full_Name")
)
master_df.head()

Rank,Player,Age,Team_Abbreviation,Games,Minutes_Per_Game,Points_Per_Game,Total_Rebounds_Per_Game,Assists_Per_Game,Steals_Per_Game,Blocks_Per_Game,Field_Goal_Percentage,3-Points_Field_Goal_Percentage,Free_Throw_Percentage,Win_Shares,Win_Shares_Per_48_Minutes,Team_Full_Name
i64,str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
1,"""Tim Duncan""",26,"""SAS""",81,39.3,23.3,12.9,3.9,0.7,2.9,0.513,0.273,0.71,16.5,0.248,"""San Antonio Spurs"""
2,"""Kevin Garnett""",26,"""MIN""",82,40.5,23.0,13.4,6.0,1.4,1.6,0.502,0.282,0.751,15.6,0.225,"""Minnesota Timberwolves"""
3,"""Kobe Bryant""",24,"""LAL""",82,41.5,30.0,6.9,5.9,2.2,0.8,0.451,0.383,0.843,14.9,0.21,"""Los Angeles Lakers"""
4,"""Tracy McGrady""",23,"""ORL""",75,39.4,32.1,6.5,5.5,1.7,0.8,0.457,0.386,0.793,16.1,0.262,"""Orlando Magic"""
5,"""Shaquille O'Neal""",30,"""LAL""",67,37.8,27.5,11.1,3.1,0.6,2.4,0.574,null,0.622,13.2,0.25,"""Los Angeles Lakers"""


### Make sure Player Name has no * or . in their name

In [164]:
master_df = master_df.with_columns(
    pl.col("Player")
    .str.replace_all(".", "", literal=True)
    .str.replace_all("*", "", literal=True)
)

### Check for Duplicated Names and Proper Name Conventions

In [165]:
sorted(master_df["Player"].unique().to_list())

['Al Jefferson',
 'Allen Iverson',
 "Amar'e Stoudemire",
 'Andrei Kirilenko',
 'Antawn Jamison',
 'Anthony Davis',
 'Anthony Edwards',
 'Anthony Mason',
 'Baron Davis',
 'Ben Simmons',
 'Ben Wallace',
 'Blake Griffin',
 'Brandon Roy',
 'Cade Cunningham',
 'Carlos Boozer',
 'Carmelo Anthony',
 'Chauncey Billups',
 'Chris Bosh',
 'Chris Paul',
 'Chris Webber',
 'Damian Lillard',
 'David Lee',
 'David Robinson',
 "De'Aaron Fox",
 'DeMar DeRozan',
 'Deron Williams',
 'Derrick Rose',
 'Devin Booker',
 'Dirk Nowitzki',
 'Domantas Sabonis',
 'Donovan Mitchell',
 'Draymond Green',
 'Dwight Howard',
 'Dwyane Wade',
 'Elton Brand',
 'Evan Mobley',
 'Gary Payton',
 'Giannis Antetokounmpo',
 'Gilbert Arenas',
 'Goran Dragić',
 'Isaiah Thomas',
 'Ja Morant',
 'Jalen Brunson',
 'Jamal Mashburn',
 'James Harden',
 'Jason Kidd',
 'Jayson Tatum',
 "Jermaine O'Neal",
 'Jerry Stackhouse',
 'Jimmy Butler',
 'Joakim Noah',
 'Joe Johnson',
 'Joel Embiid',
 'John Stockton',
 'John Wall',
 'Julius Randle',
 '

### Check for Nulls

There are no nulls so we are good here

In [166]:
master_df.null_count()

Rank,Player,Age,Team_Abbreviation,Games,Minutes_Per_Game,Points_Per_Game,Total_Rebounds_Per_Game,Assists_Per_Game,Steals_Per_Game,Blocks_Per_Game,Field_Goal_Percentage,3-Points_Field_Goal_Percentage,Free_Throw_Percentage,Win_Shares,Win_Shares_Per_48_Minutes,Team_Full_Name
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,9,0,0,0,0


### Save the Data before we aggregated it


In [167]:
dirpath = Path("data/raw_data/MVP_Voting/")
dirpath.mkdir(parents=True, exist_ok=True)
filepath = dirpath / "NBA_MVP_Voting_Data_cleaned_no_agg.csv"
if not filepath.exists():
    master_df.write_csv(filepath, separator=",")

### Aggregate Data to get the avg stats per player for all their MVP season ranking

In [168]:
master_df.columns

['Rank',
 'Player',
 'Age',
 'Team_Abbreviation',
 'Games',
 'Minutes_Per_Game',
 'Points_Per_Game',
 'Total_Rebounds_Per_Game',
 'Assists_Per_Game',
 'Steals_Per_Game',
 'Blocks_Per_Game',
 'Field_Goal_Percentage',
 '3-Points_Field_Goal_Percentage',
 'Free_Throw_Percentage',
 'Win_Shares',
 'Win_Shares_Per_48_Minutes',
 'Team_Full_Name']

In [169]:
agg_master_df = master_df.group_by("Player").agg(
    pl.len().alias("Count"),
    (pl.col("Rank") == 1).sum().alias("MVP_Won"),
    pl.col("Minutes_Per_Game").mean().alias("AVG_Minutes_Per_Game"),
    pl.col("Points_Per_Game").mean().alias("AVG_Points_Per_Game"),
    pl.col("Total_Rebounds_Per_Game").mean().alias("AVG_Total_Rebounds_Per_Game"),
    pl.col("Assists_Per_Game").mean().alias("AVG_Assists_Per_Game"),
    pl.col("Steals_Per_Game").mean().alias("AVG_Steals_Per_Game"),
    pl.col("Blocks_Per_Game").mean().alias("AVG_blocks_Per_Game"),
    pl.col("Field_Goal_Percentage").mean().alias("AVG_Field_Goal_Percentage"),
    pl.col("3-Points_Field_Goal_Percentage").mean().alias("AVG_3-Points_Field_Goal_Percentage"),
    pl.col("Free_Throw_Percentage").mean().alias("AVG_Free_Throw_Percentage"),
    pl.col("Win_Shares").mean().alias("AVG_Win_Shares"),
    pl.col("Win_Shares_Per_48_Minutes").mean().alias("AVG_Win_Shares_Per_48_Minutes"),
)
agg_master_df

Player,Count,MVP_Won,AVG_Minutes_Per_Game,AVG_Points_Per_Game,AVG_Total_Rebounds_Per_Game,AVG_Assists_Per_Game,AVG_Steals_Per_Game,AVG_blocks_Per_Game,AVG_Field_Goal_Percentage,AVG_3-Points_Field_Goal_Percentage,AVG_Free_Throw_Percentage,AVG_Win_Shares,AVG_Win_Shares_Per_48_Minutes
str,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Tim Duncan""",13,2,33.969231,19.461538,10.969231,3.1,0.738462,2.238462,0.503385,0.170385,0.697462,11.384615,0.211077
"""Anthony Mason""",1,0,40.7,16.1,9.6,3.1,1.0,0.3,0.482,null,0.781,11.6,0.171
"""Ty Lawson""",1,0,34.4,16.7,2.7,6.9,1.5,0.1,0.461,0.366,0.756,7.4,0.141
"""Kevin Love""",2,0,37.65,26.05,12.9,3.2,0.85,0.5,0.4525,0.374,0.8225,12.15,0.234
"""Anthony Davis""",4,0,35.75,26.65,10.6,2.45,1.45,2.5,0.51925,0.263,0.82025,12.45,0.24
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""David Robinson""",1,0,29.6,14.4,8.6,1.5,1.0,2.5,0.486,0.0,0.747,12.1,0.246
"""Jimmy Butler""",3,0,34.633333,21.666667,5.966667,5.4,1.866667,0.433333,0.489333,0.314667,0.846,10.066667,0.232
"""Draymond Green""",1,0,34.7,14.0,9.5,7.4,1.5,1.4,0.49,0.388,0.696,11.1,0.19


In [170]:
dirpath = Path("data/raw_data/MVP_Voting/")
dirpath.mkdir(parents=True, exist_ok=True)
filepath = dirpath / "NBA_MVP_Voting_Data_cleaned_agg.csv"
if dirpath.exists():
    agg_master_df.write_csv(filepath, separator=",")